# 07 · Retrieval evaluation
Runs the 45 in-scope eval questions through the search and measures Hit@1, Hit@3, Hit@5, MRR and
document-level Hit@5. Results are broken down by area and question type, and saved to `reports/`.

In [1]:
# ===== CELL 1: Setup =====
# Loads the eval set, the chunks, the embedding model and the vector database.

import re                                               # re: text patterns (used to tidy whitespace)
from pathlib import Path                                # Path: folders and file paths
import pandas as pd                                     # pandas: tables
import chromadb                                         # chromadb: our vector database
from sentence_transformers import SentenceTransformer   # SentenceTransformer: embeds the questions

REPORTS = Path("../reports")                            # Folder for result files
REPORTS.mkdir(exist_ok=True)                            # Create it if it doesn't exist yet

evals = pd.read_json("../evals/eval_set.jsonl", lines=True)          # The 50 questions
evals = evals[evals["doc_id"].notna()].reset_index(drop=True)        # Keep only the 45 in-scope ones (out-of-scope questions have no correct chunk)
chunks = pd.read_json("../data/processed/chunks.jsonl", lines=True)  # The 6,601 chunks

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")             # Same model used to build the database
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "   # The instruction this model expects in front of questions
collection = chromadb.PersistentClient(path="../data/chroma").get_collection("aspnetcore_docs_bge_small")   # Open the existing collection

print("In-scope questions:", len(evals))                 # Should print 45

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In-scope questions: 45


In [2]:
# ===== CELL 2: Find the correct chunks for each question =====
# A chunk is "correct" if it comes from the expected article AND contains the evidence phrase.

def norm(text):                                         # Make text easy to compare...
    return re.sub(r"\s+", " ", text).lower()            # ...by collapsing whitespace and lowercasing

chunks["norm"] = chunks["text"].map(norm)               # Normalised text of every chunk, computed once

relevant = {}                                           # question ID -> set of correct chunk IDs
for q in evals.itertuples():                            # For each question...
    mask = (chunks["doc_id"] == q.doc_id) & chunks["norm"].str.contains(norm(q.evidence), regex=False)   # ...chunks from the right article that contain the evidence
    relevant[q.id] = set(chunks.loc[mask, "chunk_id"])  # Store their IDs as a set (fast to check "is this ID in the set?")

print("Correct chunks per question:", {k: len(v) for k, v in list(relevant.items())[:8]}, "...")   # Peek at the first few

Correct chunks per question: {'F01': 1, 'F02': 1, 'F03': 1, 'F04': 2, 'F05': 1, 'F06': 1, 'F07': 1, 'F08': 1} ...


In [3]:
# ===== CELL 3: Run every question through the search and score it =====

K = 10                                                  # Fetch the top 10, so we can see how far down a correct chunk is, even beyond 5

def search_ids(question, k=K):                          # Return the IDs of the top k chunks, best first
    q = embedder.encode([QUERY_PREFIX + question], normalize_embeddings=True)   # Embed the question
    res = collection.query(query_embeddings=q.tolist(), n_results=k)            # Ask the database
    return res["ids"][0]                                # The list of chunk IDs, in rank order

rows = []                                               # One row of results per question
for q in evals.itertuples():                            # For each question...
    ids = search_ids(q.question)                        # ...get the top 10 chunk IDs
    rank = next((i for i, cid in enumerate(ids, start=1) if cid in relevant[q.id]), None)   # Position (1, 2, 3...) of the first correct chunk, or None if not in the top 10
    doc_rank = next((i for i, cid in enumerate(ids, start=1) if cid.split("::")[0] == q.doc_id), None)   # Position of the first chunk from the right article
    rows.append({
        "id": q.id, "area": q.area, "type": q.type,     # Which question
        "rank": rank,                                   # Where the correct chunk was found (None = missed)
        "hit@1": rank is not None and rank <= 1,        # True if the correct chunk was 1st
        "hit@3": rank is not None and rank <= 3,        # True if it was in the top 3
        "hit@5": rank is not None and rank <= 5,        # True if it was in the top 5
        "rr": 1 / rank if rank else 0.0,                # Reciprocal rank: 1st = 1, 2nd = 0.5, 3rd = 0.33, missed = 0
        "doc_hit@5": doc_rank is not None and doc_rank <= 5,   # True if the right ARTICLE was in the top 5
        "top1": ids[0],                                 # What the search ranked first (useful for studying misses)
    })

per_q = pd.DataFrame(rows)                              # All results as a table
print("Done:", len(per_q), "questions")

Done: 45 questions


In [4]:
# ===== CELL 4: Summary scores =====
# Averages the per-question results. True counts as 1 and False as 0, so the mean of a True/False column is a percentage.

def summarise(df):                                      # Turn per-question rows into one row of scores
    return pd.Series({
        "n": len(df),                                   # Number of questions
        "Hit@1": df["hit@1"].mean(),                    # Share with the correct chunk 1st
        "Hit@3": df["hit@3"].mean(),                    # Share with it in the top 3
        "Hit@5": df["hit@5"].mean(),                    # Share with it in the top 5
        "MRR": df["rr"].mean(),                         # Average reciprocal rank
        "Doc Hit@5": df["doc_hit@5"].mean(),            # Share with the right article in the top 5
    })

overall = summarise(per_q).to_frame("bge-small (vector only)").T     # One row: the overall scores
by_area = per_q.groupby("area").apply(summarise)       # Same scores for each area
by_type = per_q.groupby("type").apply(summarise)       # Same scores for direct vs paraphrased questions

pct = {"Hit@1": "{:.0%}", "Hit@3": "{:.0%}", "Hit@5": "{:.0%}", "MRR": "{:.3f}", "Doc Hit@5": "{:.0%}", "n": "{:.0f}"}   # How to display each column
display(overall.style.format(pct))                     # Show the overall table
display(by_area.style.format(pct))                     # Show it per area
display(by_type.style.format(pct))                     # Show it per question type

,n,Hit@1,Hit@3,Hit@5,MRR,Doc Hit@5
bge-small (vector only),45,29%,44%,60%,0.418,93%


,n,Hit@1,Hit@3,Hit@5,MRR,Doc Hit@5
area,,,,,,
blazor,15,20%,33%,53%,0.333,87%
fundamentals,15,20%,33%,47%,0.316,93%
mvc,15,47%,67%,80%,0.604,100%


,n,Hit@1,Hit@3,Hit@5,MRR,Doc Hit@5
type,,,,,,
direct,30,33%,53%,67%,0.484,97%
paraphrase,15,20%,27%,47%,0.285,87%


In [5]:
# ===== CELL 5: Study the misses, then save =====

misses = per_q[~per_q["hit@5"]].merge(evals[["id", "question", "doc_id"]], on="id")   # Questions where the correct chunk was NOT in the top 5, with their text
print(f"Missed (not in top 5): {len(misses)}\n")
for m in misses.itertuples():                           # Show each miss
    print(f"[{m.id}] {m.question}")                     # The question
    print(f"     expected article: {m.doc_id}")         # Where the answer is
    print(f"     ranked 1st:       {m.top1}")           # What the search returned instead
    print(f"     correct chunk at: {m.rank or 'not in top 10'}\n")   # How far down the correct chunk was

per_q.to_csv(REPORTS / "retrieval_per_question_bge_small.csv", index=False)   # Save the per-question details
overall.to_csv(REPORTS / "retrieval_summary.csv")       # Save the summary row (we'll add rows for each improvement later)
print("Saved to reports/")

Missed (not in top 5): 18

[F01] Which method registers a service with a scoped lifetime, and how long does a scoped service live in an MVC app?
     expected article: fundamentals/dependency-injection
     ranked 1st:       blazor/fundamentals/dependency-injection::11
     correct chunk at: nan

[F02] How do I register a service so a new instance is created per HTTP request?
     expected article: fundamentals/dependency-injection
     ranked 1st:       fundamentals/http-requests::6
     correct chunk at: nan

[F08] I changed appsettings.json while my app was running, but my IOptions<T> values didn't update. Why?
     expected article: fundamentals/configuration/options
     ranked 1st:       fundamentals/configuration/options::16
     correct chunk at: 8.0

[F10] Where are static files stored by default, and which call enables serving them?
     expected article: fundamentals/static-files
     ranked 1st:       fundamentals/static-files::35
     correct chunk at: 6.0

[F11] How do I 

In [6]:
# ===== CELL 6: A reusable evaluator =====
# Scores any search function the same way as Cell 3, so different setups can be compared fairly.

import time                                             # time: to measure speed per query
import numpy as np                                      # numpy: fast number arrays (used for sorting scores)

def evaluate(name, search_fn):                          # name: label for the table; search_fn: any function question -> ranked chunk IDs
    rows = []                                           # One row per question
    for q in evals.itertuples():                        # For each of the 45 questions...
        ids = search_fn(q.question)                     # ...run this setup's search
        rank = next((i for i, cid in enumerate(ids, start=1) if cid in relevant[q.id]), None)   # Position of the first correct chunk (None = missed)
        doc_rank = next((i for i, cid in enumerate(ids, start=1) if cid.split("::")[0] == q.doc_id), None)   # Position of the first chunk from the right article
        rows.append({
            "id": q.id, "area": q.area, "type": q.type, "rank": rank,
            "hit@1": rank is not None and rank <= 1,
            "hit@3": rank is not None and rank <= 3,
            "hit@5": rank is not None and rank <= 5,
            "rr": 1 / rank if rank else 0.0,
            "doc_hit@5": doc_rank is not None and doc_rank <= 5,
        })
    per = pd.DataFrame(rows)                            # Per-question results
    return per, summarise(per).rename(name)             # Return the details and one summary row (summarise() comes from Cell 4)

In [7]:
# ===== CELL 7: Keyword search (BM25) and hybrid search =====
# BM25 is classic keyword search: it scores chunks by how many of the question's words they contain,
# giving more weight to rare words (like "IOptionsSnapshot") than common ones (like "how").

from rank_bm25 import BM25Okapi                         # BM25Okapi: a standard implementation of BM25

def tokenize(text):                                     # Split text into lowercase words
    return re.findall(r"\w+", text.lower())             # \w+ = runs of letters, digits or underscores

bm25 = BM25Okapi([tokenize(t) for t in chunks["embed_text"]])   # Build the keyword index over every chunk (header + text)
chunk_ids = chunks["chunk_id"].tolist()                 # Chunk IDs in the same order as the index

def bm25_ids(question, k=K):                            # Keyword search: return the top k chunk IDs
    scores = bm25.get_scores(tokenize(question))        # Score every chunk against the question's words
    top = np.argsort(scores)[::-1][:k]                  # Positions of the k highest scores (argsort sorts low->high, [::-1] reverses it)
    return [chunk_ids[i] for i in top]                  # Turn positions into chunk IDs

def hybrid_ids(question, k=K, pool=50, c=60):           # Hybrid: merge vector and keyword rankings
    scores = {}                                         # chunk ID -> combined score
    for ranking in (search_ids(question, pool), bm25_ids(question, pool)):   # Take the top 50 from each search
        for r, cid in enumerate(ranking, start=1):      # r = the chunk's position in that list
            scores[cid] = scores.get(cid, 0) + 1 / (c + r)   # Reciprocal Rank Fusion: ranking high in EITHER list earns points; high in BOTH earns the most
    return sorted(scores, key=scores.get, reverse=True)[:k]   # The k chunks with the highest combined score

In [8]:
# ===== CELL 8: The reranker =====
# A cross-encoder reads the question and a chunk TOGETHER and scores how well the chunk answers it.
# It's too slow to run on all 6,601 chunks, so we use it to re-order the top 30 candidates from a fast search.

from sentence_transformers import CrossEncoder          # CrossEncoder: loads reranking models

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")   # A small, fast reranker (~90 MB download the first time)
text_by_id = dict(zip(chunks["chunk_id"], chunks["embed_text"]))  # Quick lookup: chunk ID -> its text (header + passage)

def rerank(question, candidate_ids, k=K):               # Re-order candidates by the reranker's score
    scores = reranker.predict([(question, text_by_id[c]) for c in candidate_ids])   # Score each (question, chunk) pair
    order = np.argsort(scores)[::-1][:k]                # Positions of the k best scores
    return [candidate_ids[i] for i in order]            # The candidates in their new order

def vector_rerank_ids(question, k=K):                   # Vector search for 30 candidates, then rerank
    return rerank(question, search_ids(question, 30), k)

def hybrid_rerank_ids(question, k=K):                   # Hybrid search for 30 candidates, then rerank
    return rerank(question, hybrid_ids(question, 30), k)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

d:\AI Projects\aspnetcore-rag-eval\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lenovo\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\AI Projects\aspnetcore-rag-eval\.venv\Lib\site-packages\huggingface_hub\file_download.p

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [9]:
# ===== CELL 9: Compare all five setups =====
# Takes a few minutes: the reranker setups are the slow ones on a laptop CPU.

configs = {                                             # Label -> search function
    "Vector (bge-small)": search_ids,
    "Keyword (BM25)": bm25_ids,
    "Hybrid (vector + BM25)": hybrid_ids,
    "Vector + reranker": vector_rerank_ids,
    "Hybrid + reranker": hybrid_rerank_ids,
}

summaries, per_config = [], {}                          # Summary rows, and per-question details for each setup
for name, fn in configs.items():                        # Run every setup
    start = time.time()                                 # Start the timer
    per, summary = evaluate(name, fn)                   # Score it on all 45 questions
    summary["sec/query"] = (time.time() - start) / len(evals)   # Average seconds per question
    per_config[name] = per                              # Keep the details
    summaries.append(summary)                           # Keep the summary row
    print(f"{name}: done")                              # Progress message

comparison = pd.DataFrame(summaries)                    # All summary rows as one table
comparison.to_csv(REPORTS / "retrieval_comparison.csv") # Save it for the README
display(comparison.style.format({**pct, "sec/query": "{:.2f}"}))   # Show it (pct is the format dictionary from Cell 4)

Vector (bge-small): done
Keyword (BM25): done
Hybrid (vector + BM25): done
Vector + reranker: done
Hybrid + reranker: done


,n,Hit@1,Hit@3,Hit@5,MRR,Doc Hit@5,sec/query
Vector (bge-small),45,29%,44%,60%,0.418,93%,0.05
Keyword (BM25),45,24%,47%,58%,0.384,87%,0.04
Hybrid (vector + BM25),45,36%,58%,64%,0.483,87%,0.11
Vector + reranker,45,38%,53%,73%,0.505,91%,4.46
Hybrid + reranker,45,40%,60%,78%,0.544,93%,4.51


In [10]:
# ===== CELL 10: Where each setup helps =====

print("Hit@5 by question type:")                        # Direct vs paraphrased, for each setup
display(pd.DataFrame({n: p.groupby("type")["hit@5"].mean() for n, p in per_config.items()}).T.style.format("{:.0%}"))

print("Rank of the correct chunk for the 18 original misses (blank = not in top 10):")
ranks = pd.DataFrame({n: p.set_index("id")["rank"] for n, p in per_config.items()})   # Question x setup table of ranks
display(ranks.loc[misses["id"]])                        # Only the questions the first setup missed (misses comes from Cell 5)

Hit@5 by question type:


type,direct,paraphrase
Vector (bge-small),67%,47%
Keyword (BM25),60%,53%
Hybrid (vector + BM25),70%,53%
Vector + reranker,77%,67%
Hybrid + reranker,80%,73%


Rank of the correct chunk for the 18 original misses (blank = not in top 10):


,Vector (bge-small),Keyword (BM25),Hybrid (vector + BM25),Vector + reranker,Hybrid + reranker
id,,,,,
F01,NaN,3.0,6.0,9.0,9.0
F02,NaN,NaN,NaN,NaN,NaN
F08,8.0,NaN,NaN,5.0,5.0
F10,6.0,4.0,3.0,2.0,2.0
F11,NaN,1.0,6.0,NaN,1.0
F13,9.0,NaN,2.0,4.0,4.0
F14,NaN,NaN,NaN,NaN,NaN
F15,NaN,NaN,NaN,1.0,1.0
B02,10.0,3.0,3.0,2.0,2.0
